# 03 — Podela na trening i test skup

Ova sveska predstavlja **principijelno najvažniji korak** u projektu:
podelu podataka na trening (train) i test skup.

**Zašto je ovo najkritičniji korak:** sve odluke koje donosimo u
narednim sveskama (detekcija outliera, imputacija, feature engineering,
skaliranje) moraju biti bazirane **isključivo na trening skupu**. Test
skup se otvara samo jednom — na kraju, za finalnu evaluaciju modela.

Ovaj princip sprečava **curenje informacija** (*data leakage*) i garantuje
da su rezultati evaluacije pošteni i reprezentativni za rad modela na
budućim, do sada neviđenim podacima.

## 1. Učitavanje podataka i strukturne popravke

Učitavamo skup i vršimo dve strukturne popravke koje smo najavili u
prethodnim sveskama:

1. **Konverzija `TotalCharges` u numerički tip** — jer je 11 vrednosti
   sadržalo prazan razmak, cela kolona je bila prepoznata kao tekst.
2. **Uklanjanje 11 redova sa `NaN` u `TotalCharges`** — radi se o
   korisnicima sa `tenure = 0` (novopridošli, još nisu naplaćeni). Broj
   redova je zanemarljiv (0.16%), a njihova specifičnost bi mogla da
   unese šum u model.

**Napomena:** ove popravke su strukturne — ne zavise od distribucije
podataka, već su posledica objektivnih problema u sirovom fajlu. Zato
ih radimo pre split-a bez rizika od *data leakage*.

In [1]:
import pandas as pd
import numpy as np

# train_test_split je funkcija za podelu podataka na trening i test skup.
from sklearn.model_selection import train_test_split

# Podešavanje pandas-a: prikazuj sve kolone bez skraćivanja.
pd.set_option("display.max_columns", None)

# Učitavamo CSV.
df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")


df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Strukturna popravka 2: brisanje 11 redova sa NaN u TotalCharges.
# Ovo su korisnici sa tenure=0 (novi, još nisu naplaćeni).
df = df.dropna(subset=["TotalCharges"])

# .reset_index(drop=True) resetuje indekse (redne brojeve) redova.
df = df.reset_index(drop=True)

print(f"Nakon strukturnih popravki: {df.shape[0]} redova × {df.shape[1]} kolona")

Nakon strukturnih popravki: 7032 redova × 21 kolona


## 2. Razdvajanje atributa (X) od ciljne promenljive (y)

Pre podele na train/test, moramo razdvojiti:
- **X** — atributi (features) na osnovu kojih model uči i predviđa
- **y** — ciljna promenljiva (target) koju predviđamo (`Churn`)

Takođe izbacujemo kolonu `customerID`, jer je to jedinstveni identifikator
korisnika koji nema prediktivnu vrednost — svaki korisnik ima različit ID,
pa model ne može da nauči obrazac iz njega.

**Konvencija imenovanja:**
- Veliko slovo `X` — koristi se za matricu atributa (više kolona)
- Malo slovo `y` — koristi se za vektor ciljne promenljive (jedna kolona)



In [2]:
# Definišemo ciljnu promenljivu (y) — kolonu koju predviđamo.
y = df["Churn"]

# Definišemo matricu atributa (X) — sve kolone OSIM Churn-a i customerID-a.
# df.drop(columns=[...]) vraća novi DataFrame bez tih kolona.
# customerID izbacujemo jer nema prediktivnu vrednost.
# Churn izbacujemo jer je to ciljna promenljiva (ne sme biti među atributima).
X = df.drop(columns=["customerID", "Churn"])

# Proveravamo dimenzije.
print(f"X (atributi): {X.shape[0]} redova × {X.shape[1]} kolona")
print(f"y (ciljna promenljiva): {y.shape[0]} vrednosti")
print(f"\nAtributi u X: {list(X.columns)}")

X (atributi): 7032 redova × 19 kolona
y (ciljna promenljiva): 7032 vrednosti

Atributi u X: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 3. Podela na trening i test skup

Sada delimo podatke na dva skupa:
- **Trening skup (80%)** — koristimo za sve dalje analize, transformacije i
  treniranje modela
- **Test skup (20%)** — zaključavamo do samog kraja projekta; koristi se
  isključivo za finalnu evaluaciju

**Parametri koje koristimo pri podeli:**

- `test_size=0.2` — 20% podataka ide u test skup
- `stratify=y` — **stratifikacija po ciljnoj promenljivoj**. Garantuje da
  proporcija klasa (Yes/No) bude ista u train i test skupu. Ovo je ključno
  za nebalansirane skupove kao što je naš (~73% : ~27%).
- `random_state=42` — fiksira „seme" slučajnog procesa. Svaki put kad se
  kod pokrene, dobija se **ista** podela. Ovo je neophodno za ponovljivost
  eksperimenata.
- `shuffle=True` — pre podele, redovi se izmešaju. Sprečava da eventualni
  redosled u sirovim podacima utiče na podelu (npr. da svi noviji korisnici
  budu na kraju).

In [3]:
# Delimo X i y na train i test skup jednim pozivom funkcije.
# train_test_split vraća 4 objekta u fiksnom redosledu:
#   X_train, X_test, y_train, y_test
# Redosled je uvek ovakav — X pre y, train pre test.
X_train, X_test, y_train, y_test = train_test_split(
    X,                    # matrica atributa
    y,                    # ciljna promenljiva
    test_size=0.2,        # 20% za test
    stratify=y,           # stratifikacija po ciljnoj promenljivoj
    random_state=42,      # fiksiramo slučajnost za ponovljivost
    shuffle=True          # izmešaj redove pre podele
)

# Prikazujemo dimenzije oba skupa da potvrdimo podelu.
print("Dimenzije nakon podele:")
print(f"  X_train: {X_train.shape[0]} redova x {X_train.shape[1]} kolona")
print(f"  X_test:  {X_test.shape[0]} redova x {X_test.shape[1]} kolona")
print(f"  y_train: {y_train.shape[0]} vrednosti")
print(f"  y_test:  {y_test.shape[0]} vrednosti")

# Procentualno odnos
ukupno = X_train.shape[0] + X_test.shape[0]
print(f"\nProcenat train: {100 * X_train.shape[0] / ukupno:.2f}%")
print(f"Procenat test:  {100 * X_test.shape[0] / ukupno:.2f}%")

Dimenzije nakon podele:
  X_train: 5625 redova x 19 kolona
  X_test:  1407 redova x 19 kolona
  y_train: 5625 vrednosti
  y_test:  1407 vrednosti

Procenat train: 79.99%
Procenat test:  20.01%


## 4. Provera stratifikacije

Nakon podele, obavezno proveravamo da je stratifikacija **stvarno radila** —
da proporcija klasa `Churn` bude ista (ili gotovo identična) u train i test
skupu.

Ako se procenti razlikuju značajno, nešto nije u redu i moramo istražiti.

In [4]:
# Procentualna raspodela Churn u originalnom skupu, train skupu i test skupu.
# normalize=True vraća proporcije umesto apsolutnih brojeva.
print("Proporcije Churn klasa:")
print("\nOriginalni skup:")
print(y.value_counts(normalize=True) * 100)

print("\nTrain skup:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest skup:")
print(y_test.value_counts(normalize=True) * 100)

Proporcije Churn klasa:

Originalni skup:
Churn
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64

Train skup:
Churn
No     73.422222
Yes    26.577778
Name: proportion, dtype: float64

Test skup:
Churn
No     73.418621
Yes    26.581379
Name: proportion, dtype: float64


### Rezultat provere

Proporcije klase `Churn` su gotovo identične u sva tri skupa (originalni,
train, test) — sve tri pokazuju odnos ~73.4% : ~26.6%. **Stratifikacija je
uspela** i oba skupa su reprezentativna po ciljnoj promenljivoj.

Ovo znači da će evaluacija modela na test skupu biti poštena — proporcija
klasa koju model vidi tokom evaluacije odgovara stvarnoj proporciji u populaciji.

## 5. Čuvanje train i test skupova u fajlove

Kako naredne sveske (`04`, `05`, `06`, modeli) ne bi morale svaki put da
ponavljaju učitavanje, konverziju i split, sačuvaćemo trenutno stanje
train i test skupova u fajlove u folderu `data/processed/`.

**Prednosti ovakvog pristupa:**
- Svaka sledeća sveska počinje sa **istim, poznatim stanjem** podataka
- Ne postoji rizik da se u različitim sveskama neka transformacija
  primeni drugačije
- Ušteda vremena — ne pokreće se svaka sveska od nule
- Reproduktivnost — ako neko klonira repo, može pokrenuti bilo koji
  notebook nezavisno

**Format fajla:** biramo **CSV** — čitljiv, univerzalan, i lako se otvara
u bilo kom alatu (Excel, Python, R). Alternativa bi bio Parquet (brži
i manji), ali za naš skup od 7.032 reda CSV je sasvim dovoljan.

Čuvamo **četiri fajla**:
- `X_train.csv` i `y_train.csv` — trening podaci
- `X_test.csv` i `y_test.csv` — test podaci (ne otvaramo do kraja projekta)

In [5]:
# Definišemo putanju do foldera gde ćemo čuvati fajlove.
# "../data/processed/" znači: izađi iz notebooks/, uđi u data/processed/.
putanja = "../data/processed/"

# Čuvamo svaki od 4 objekta kao poseban CSV fajl.
# index=False znači: ne čuvaj indekse (redne brojeve) kao dodatnu kolonu u CSV-u.
# Ako bismo ostavili default (index=True), pri sledećem učitavanju bismo dobili
# suvišnu kolonu "Unnamed: 0" — svima na nervima i beskorisnu.
X_train.to_csv(putanja + "X_train.csv", index=False)
X_test.to_csv(putanja + "X_test.csv", index=False)
y_train.to_csv(putanja + "y_train.csv", index=False)
y_test.to_csv(putanja + "y_test.csv", index=False)

print("Fajlovi uspešno sačuvani u folder 'data/processed/':")
print("  - X_train.csv")
print("  - X_test.csv")
print("  - y_train.csv")
print("  - y_test.csv")

Fajlovi uspešno sačuvani u folder 'data/processed/':
  - X_train.csv
  - X_test.csv
  - y_train.csv
  - y_test.csv


## 6. Verifikacija: učitavanje fajlova nazad

Kao završna provera, učitavamo sačuvane fajlove nazad i uveravamo se da
su dimenzije očuvane. Ovo je važno jer sledeće sveske startuju baš iz
ovih fajlova

In [6]:
# Učitavamo fajlove nazad da proverimo da je čuvanje bilo korektno.
X_train_check = pd.read_csv(putanja + "X_train.csv")
X_test_check = pd.read_csv(putanja + "X_test.csv")
y_train_check = pd.read_csv(putanja + "y_train.csv")
y_test_check = pd.read_csv(putanja + "y_test.csv")

print("Dimenzije nakon učitavanja iz fajlova:")
print(f"  X_train: {X_train_check.shape[0]} × {X_train_check.shape[1]}")
print(f"  X_test:  {X_test_check.shape[0]} × {X_test_check.shape[1]}")
print(f"  y_train: {y_train_check.shape[0]}")
print(f"  y_test:  {y_test_check.shape[0]}")

# Provera identičnosti — dimenzije treba da se poklope sa originalnim.
provere = [
    X_train.shape == X_train_check.shape,
    X_test.shape == X_test_check.shape,
    y_train.shape[0] == y_train_check.shape[0],
    y_test.shape[0] == y_test_check.shape[0],
]

if all(provere):
    print("\nSve dimenzije se poklapaju. Fajlovi su korektno sačuvani.")
else:
    print("\nUPOZORENJE: dimenzije se ne poklapaju — istražiti problem!")

Dimenzije nakon učitavanja iz fajlova:
  X_train: 5625 × 19
  X_test:  1407 × 19
  y_train: 5625
  y_test:  1407

Sve dimenzije se poklapaju. Fajlovi su korektno sačuvani.


---

## 7. Zaključak

U ovoj svesci smo uspešno izvršili **principijelno najvažniji korak**
projekta — podelu podataka na trening i test skup.

**Šta smo uradili:**

1. **Strukturne popravke** — konverzija `TotalCharges` u numerički tip
   i uklanjanje 11 redova sa nedostajućim vrednostima
2. **Razdvajanje X i y** — atributi (19 kolona) odvojeni od ciljne
   promenljive (`Churn`)
3. **Stratifikovana podela** — 80% train, 20% test, sa očuvanjem
   proporcija klasa u oba skupa
4. **Verifikacija** — proporcije `Churn` su praktično identične u train
   i test skupu (~73.4% / ~26.6%)
5. **Čuvanje** — sva četiri objekta (X_train, X_test, y_train, y_test)
   sačuvana u `data/processed/` folder

**Dimenzije nakon podele:**
- Train: 5.625 korisnika (80%)
- Test: 1.407 korisnika (20%)

**Ključni princip koji sledimo od ovog trenutka:**

> Sve odluke, analize i transformacije od sada rade se **isključivo na
> trening skupu**. Test skup se ne otvara, ne analizira, i ne koristi za
> bilo kakvo računanje statistika — do samog kraja projekta, kada se
> koristi za finalnu evaluaciju modela.

